In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import vectorbt as vbt

In [ ]:
df = pd.read_csv('data/Data.csv', header=[0, 1], index_col=0)
df.head()

In [ ]:
for stock in df.columns.get_level_values(0).unique():
    close = df[(stock, 'Close')]
    print(f'{stock}: {close.isna().sum()} NaNs, {(close == 0).sum()} zeros')

In [ ]:
df_clean = df.copy()

for stock in df_clean.columns.get_level_values(0).unique():
    for field in ['Close', 'Open', 'High', 'Low']:
        s = df_clean[(stock, field)].replace(0, np.nan).ffill().bfill()
        df_clean[(stock, field)] = s

close = pd.DataFrame({
    stock: df_clean[(stock, 'Close')]
    for stock in df_clean.columns.get_level_values(0).unique()
})

print(close.isna().any().any(), (close == 0).any().any())
close.head()

In [ ]:
close.plot(figsize=(10, 5))
plt.xlabel('Time')
plt.ylabel('Price')
plt.tight_layout()
plt.savefig('results/01_prices.png', dpi=120)
plt.show()

In [ ]:
def compute_rsi(s, period=14):
    delta = s.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(50)

rsi = close.apply(lambda s: compute_rsi(s, 14))
rsi.head(20)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
close['Stock_A'].iloc[:500].plot(ax=axes[0])
axes[0].set_ylabel('Price')
rsi['Stock_A'].iloc[:500].plot(ax=axes[1], color='orange')
axes[1].axhline(70, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(30, color='green', linestyle='--', alpha=0.5)
axes[1].set_ylabel('RSI')
plt.tight_layout()
plt.savefig('results/02_rsi.png', dpi=120)
plt.show()

In [ ]:
def compute_volatility(s, period=14):
    return s.pct_change().rolling(window=period, min_periods=period).std()

vol = close.apply(lambda s: compute_volatility(s, 14))
vol.head(20)

In [ ]:
L, H, W = 30, 70, 20

vol_ma = vol.rolling(window=W, min_periods=W).mean()
vol_ok = vol < vol_ma

entries = ((rsi < L) & vol_ok).fillna(False)
exits = (rsi > H).fillna(False)

print(entries.sum())
print(exits.sum())

In [ ]:
pf = vbt.Portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    fees=0.0,
    init_cash=10_000.0,
    freq='1min',
)

summary = pd.DataFrame({
    'total_return': pf.total_return(),
    'sharpe_ratio': pf.sharpe_ratio(),
    'max_drawdown': pf.max_drawdown(),
    'n_trades': pf.trades.count(),
})
summary

In [ ]:
pf.value().plot(figsize=(10, 5))
plt.xlabel('Time')
plt.ylabel('Portfolio Value')
plt.tight_layout()
plt.savefig('results/03_equity_curves.png', dpi=120)
plt.show()